# Analiza puli 1.68M kandydatów

Cel: zrozumieć rozkład domen, duplikatów URL, pred_proba -- podjąć decyzję o strategii próbkowania do full scan.

In [1]:
import polars as pl
import numpy as np

df = pl.read_parquet('../stage4/results/predictions_50M.parquet')
print(f'Total candidates: {len(df):,}')
print(f'Columns: {df.columns}')
df.head()

Total candidates: 1,675,572
Columns: ['uid', 'url', 'pred_proba', 'score']


uid,url,pred_proba,score
str,str,f64,f64
"""5dce984bfbad1215f990f569cf3de6…","""http://res.cngoldres.com/uploa…",1.0,0.0
"""4a23432cc036ee88467096a3cf073e…","""http://www.pingju365.com/ad/wp…",1.0,0.0
"""c405e6d01e49ffe6b28d516468cb45…","""https://thumbor.forbes.com/thu…",1.0,0.0
"""6510adbf45ffbd11c5d7938c782b83…","""https://ourworldindata.org/upl…",1.0,-1.0
"""b982dfd15448b0f92fd6c463767281…","""https://www.medrxiv.org/conten…",1.0,0.0


In [2]:
# Normalize URL: strip protocol (http/https), strip www., strip trailing slash
# So that http://www.example.com/map.jpg == https://example.com/map.jpg
df = df.with_columns(
    pl.col('url')
      .str.replace(r'^https?://', '')
      .str.replace(r'^www\.', '')
      .str.strip_chars('/')
      .alias('url_norm')
)

# Extract domain from normalized URL
df = df.with_columns(
    pl.col('url_norm').str.extract(r'^([^/]+)', 1).alias('domain')
)

print(f'Total records:           {len(df):,}')
print(f'Unique raw URLs:         {df.select("url").n_unique():,}')
print(f'Unique normalized URLs:  {df.select("url_norm").n_unique():,}')
print(f'Duplicate URLs (norm):   {len(df) - df.select("url_norm").n_unique():,}')
print(f'Unique domains:          {df.select("domain").n_unique():,}')


Unique domains: 153,310
Unique URLs: 1,355,812
Duplicate URLs: 319,760


## 1. Duplikaty URL

Najpierw sprawdzamy ile URL-ów się powtarza. Duplikaty = ten sam obraz w wielu rekordach MapPool.

In [3]:
url_counts = df.group_by('url_norm').agg(
    pl.len().alias('n_duplicates'),
    pl.col('domain').first().alias('domain'),
    pl.col('url').first().alias('sample_url'),
    pl.col('pred_proba').mean().alias('mean_proba'),
).sort('n_duplicates', descending=True)

print('=== URL DUPLICATION (after normalization) ===')
print(f'URLs appearing 1x:    {url_counts.filter(pl.col("n_duplicates") == 1).height:,}')
print(f'URLs appearing 2-5x:  {url_counts.filter((pl.col("n_duplicates") >= 2) & (pl.col("n_duplicates") <= 5)).height:,}')
print(f'URLs appearing 6-50x: {url_counts.filter((pl.col("n_duplicates") >= 6) & (pl.col("n_duplicates") <= 50)).height:,}')
print(f'URLs appearing 50+x:  {url_counts.filter(pl.col("n_duplicates") > 50).height:,}')
print()
print(f'Total unique images (after norm): {url_counts.height:,}')
print(f'Duplicates removed: {len(df) - url_counts.height:,}')


=== URL DUPLICATION ===
URLs appearing 1x:    1,273,711
URLs appearing 2-5x:  73,481
URLs appearing 6-50x: 8,431
URLs appearing 50+x:  189

Total records if deduplicated to unique URLs: 1,355,812 (saving 319,760 duplicates)


In [4]:
# Top 30 most duplicated URLs (normalized)
print('=== TOP 30 MOST DUPLICATED URLs ===')
for row in url_counts.head(30).iter_rows(named=True):
    print(f'  {row["n_duplicates"]:>5}x  {row["domain"]:<35}  {row["sample_url"][:90]}')


=== TOP 30 MOST DUPLICATED URLs ===
  104083x  ville-data.com                       http://ville-data.com/image/carte-france.jpg
   2484x  www.policeofficer.education          https://www.policeofficer.education/wp-content/uploads/2017/03/police-officer-salary-by-st
   1279x  cdn4.uvnimg.com                      https://cdn4.uvnimg.com/dims4/default/7859c8e/2147483647/thumbnail/400x250%3E/quality/75/?
    938x  cilentano.it                         https://cilentano.it/wp-content/uploads/2020/04/drone.jpg
    830x  cilentano.it                         https://cilentano.it/wp-content/uploads/2020/04/dronemondo.jpg
    532x  i0.kym-cdn.com                       http://i0.kym-cdn.com/photos/images/facebook/000/714/361/d96.png
    517x  i.servimg.com                        https://i.servimg.com/u/f53/16/37/80/01/sans_201.png
    493x  cs5.pikabu.ru                        http://cs5.pikabu.ru/post_img/2015/12/27/9/1451229603183192180.jpg
    428x  cdn.gobankingrates.com               https:/

### Decyzja: deduplikować URL-e przed samplingiem?

Jeśli ten sam URL pojawia się 100x, nie ma sensu go pobierać 100 razy. Warto deduplikować.

In [5]:
# Deduplicate: keep one record per unique normalized URL (highest pred_proba)
df_dedup = df.sort('pred_proba', descending=True).unique(subset=['url_norm'], keep='first')
print(f'After URL deduplication: {len(df_dedup):,} (was {len(df):,}, removed {len(df) - len(df_dedup):,})')
print(f'Unique domains: {df_dedup.select("domain").n_unique():,}')


After URL deduplication: 1,355,812 (was 1,675,572, removed 319,760)
Unique domains: 153,310


## 2. Analiza domen

Po deduplikacji URL -- ile obrazów per domena?

In [6]:
domains = df_dedup.group_by('domain').agg(
    pl.len().alias('count'),
    pl.col('pred_proba').mean().alias('mean_proba'),
    pl.col('pred_proba').median().alias('median_proba'),
    pl.col('pred_proba').min().alias('min_proba'),
    pl.col('pred_proba').max().alias('max_proba'),
).sort('count', descending=True)

print('=== DOMAIN SIZE DISTRIBUTION (after URL dedup) ===')
for lo, hi, label in [(1,1,'1 image'), (2,10,'2-10'), (11,50,'11-50'), (51,100,'51-100'), 
                       (101,500,'101-500'), (501,1000,'501-1k'), (1001,10000,'1k-10k'), (10001,999999,'10k+')]:
    n = domains.filter((pl.col('count') >= lo) & (pl.col('count') <= hi)).height
    imgs = domains.filter((pl.col('count') >= lo) & (pl.col('count') <= hi))['count'].sum()
    print(f'  {label:>10}: {n:>6,} domains, {imgs:>10,} images')

=== DOMAIN SIZE DISTRIBUTION (after URL dedup) ===
     1 image: 88,002 domains,     88,002 images
        2-10: 55,610 domains,    193,360 images
       11-50:  7,656 domains,    159,463 images
      51-100:  1,052 domains,     73,079 images
     101-500:    815 domains,    164,327 images
      501-1k:     86 domains,     57,885 images
      1k-10k:     78 domains,    201,585 images
        10k+:     11 domains,    418,111 images


In [7]:
# Top 50 domains -- to manually review
print('=== TOP 50 DOMAINS (after URL dedup) ===')
print(f'{"#":>3}  {"Domain":<45}  {"Count":>7}  {"Mean p":>7}  {"Sample URL"}')
print('-' * 130)

for i, row in enumerate(domains.head(50).iter_rows(named=True), 1):
    # Get a sample URL for this domain
    sample_url = df_dedup.filter(pl.col('domain') == row['domain']).select('url').head(1).item()
    print(f'{i:>3}  {row["domain"]:<45}  {row["count"]:>7,}  {row["mean_proba"]:>7.3f}  {sample_url[:60]}')

=== TOP 50 DOMAINS (after URL dedup) ===
  #  Domain                                           Count   Mean p  Sample URL
----------------------------------------------------------------------------------------------------------------------------------
  1  www.namespedia.com                             206,215    0.913  http://www.namespedia.com/img/Germany/Remers.jpg
  2  ipad.fas.usda.gov                               56,050    0.777  https://ipad.fas.usda.gov/rssiws/images/us/temp/us0l_tmx7_20
  3  app.23degrees.io                                39,180    0.932  https://app.23degrees.io/services/image/v1/getPreviewFromSlu
  4  i0.wp.com                                       21,322    0.832  https://i0.wp.com/municipiosur.com/wp-content/uploads/2020/0
  5  upload.wikimedia.org                            17,640    0.845  https://upload.wikimedia.org/wikipedia/commons/thumb/7/75/Pe
  6  droughtmonitor.unl.edu                          14,139    0.892  https://droughtmonitor.unl.edu/dat

### Ręczna weryfikacja domen

Poniższa komórka generuje listę top domen do ręcznego przejrzenia.
Oznacz domeny do wykluczenia (not_maps, templates, duplicates).

In [9]:
# MANUAL REVIEW: edit this list after inspecting URLs
# Mark domains to EXCLUDE from full scan

EXCLUDE_DOMAINS = [
    'ville-data.com',        # 6 unique URLs, 104k duplicates -- no value
    'files.airnowtech.org',  # air quality maps -- ordinal categories, should fail F3.2
    'droughtmonitor.unl.edu',# drought maps -- ordinal categories
]

# Uncomment domains above after reviewing their URLs
# Then re-run cells below to see impact on pool size

In [10]:
# Show 5 sample URLs for each top 30 domain (for manual review)
for row in domains.head(30).iter_rows(named=True):
    domain = row['domain']
    count = row['count']
    samples = df_dedup.filter(pl.col('domain') == domain).sample(min(5, count), seed=42)
    
    print(f'\n=== {domain} ({count:,} images, mean_p={row["mean_proba"]:.3f}) ===')
    for s in samples.iter_rows(named=True):
        print(f'  p={s["pred_proba"]:.3f}  {s["url"][:110]}')


=== www.namespedia.com (206,215 images, mean_p=0.913) ===
  p=0.977  http://www.namespedia.com/img/Switzerland/Schaffert.jpg
  p=0.918  http://www.namespedia.com/img/France/Benredjem.jpg
  p=0.971  https://www.namespedia.com/img/France/Paillat.jpg
  p=0.978  http://www.namespedia.com/img/USA/Hjerstedt.jpg
  p=0.948  http://www.namespedia.com/img/USA/gannon.jpg

=== ipad.fas.usda.gov (56,050 images, mean_p=0.777) ===
  p=0.694  https://ipad.fas.usda.gov/rssiws/images/world/temp/world0hdd7_201808_26_s_new.png
  p=0.650  https://ipad.fas.usda.gov/rssiws/images/chile/pasg/chile0modis_1pasg202006_30_STD.png
  p=0.928  https://ipad.fas.usda.gov/rssiws/images/umb/smap/umb0l_lalsmap7_202008_09.png
  p=0.744  https://ipad.fas.usda.gov/rssiws/images/umb/ndvi/umb0modis202102_10_STD.png
  p=0.950  https://ipad.fas.usda.gov/rssiws/images/sasia/esi/sasia0esi_12wk7_202005_03.png

=== app.23degrees.io (39,180 images, mean_p=0.932) ===
  p=0.984  https://app.23degrees.io/services/image/v1/getPreviewFr

## 3. pred_proba distribution

In [11]:
print('=== PRED_PROBA DISTRIBUTION (after URL dedup) ===')
for lo, hi in [(0.50,0.55),(0.55,0.60),(0.60,0.65),(0.65,0.70),(0.70,0.75),(0.75,0.80),
               (0.80,0.85),(0.85,0.90),(0.90,0.95),(0.95,1.01)]:
    n = df_dedup.filter((pl.col('pred_proba') >= lo) & (pl.col('pred_proba') < hi)).height
    print(f'  [{lo:.2f}, {hi:.2f}): {n:>10,}  ({n/len(df_dedup)*100:>5.1f}%)  {"█" * int(n/len(df_dedup)*100)}')

=== PRED_PROBA DISTRIBUTION (after URL dedup) ===
  [0.50, 0.55):     76,925  (  5.7%)  █████
  [0.55, 0.60):     67,748  (  5.0%)  ████
  [0.60, 0.65):     70,133  (  5.2%)  █████
  [0.65, 0.70):     72,961  (  5.4%)  █████
  [0.70, 0.75):     79,056  (  5.8%)  █████
  [0.75, 0.80):     91,544  (  6.8%)  ██████
  [0.80, 0.85):    109,698  (  8.1%)  ████████
  [0.85, 0.90):    142,053  ( 10.5%)  ██████████
  [0.90, 0.95):    237,230  ( 17.5%)  █████████████████
  [0.95, 1.01):    408,464  ( 30.1%)  ██████████████████████████████


## 4. Scenariusze próbkowania

Porównanie strategii: ile map w finalnym datasecie, jaki koszt.

In [12]:
# Assumptions from pilot
URL_ALIVE_RATE = 0.65
DOWNLOAD_RATE = 0.97
AI_PASS_RATE = 0.75
COST_PER_IMAGE = 0.002  # Haiku with caching, $/image
BUDGET_FORMAL = 1800    # USD, from mikrogrant

# Apply exclusions
df_clean = df_dedup.filter(~pl.col('domain').is_in(EXCLUDE_DOMAINS))
print(f'After exclusions: {len(df_clean):,} (excluded {len(df_dedup) - len(df_clean):,})')
print()

scenarios = [
    ('A: p>=0.70, cap 50',   0.70, 50),
    ('B: p>=0.70, cap 100',  0.70, 100),
    ('C: p>=0.60, cap 100',  0.60, 100),
    ('D: p>=0.50, cap 100',  0.50, 100),
    ('E: p>=0.70, cap 200',  0.70, 200),
    ('F: p>=0.50, cap 50',   0.50, 50),
    ('G: p>=0.80, cap 100',  0.80, 100),
    ('H: p>=0.50, no cap',   0.50, None),
]

print(f'{"Scenario":<25} {"Candidates":>12} {"Domains":>8} {"Est. alive":>11} {"Est. downloaded":>16} {"Est. maps":>10} {"Cost":>8} {"% budget":>9}')
print('-' * 110)

for name, pred_min, cap in scenarios:
    filtered = df_clean.filter(pl.col('pred_proba') >= pred_min)
    
    if cap:
        capped = filtered.group_by('domain').agg(
            pl.len().alias('n')
        ).with_columns(
            pl.when(pl.col('n') > cap).then(cap).otherwise(pl.col('n')).alias('capped')
        )
        n_cand = capped['capped'].sum()
        n_dom = capped.height
    else:
        n_cand = filtered.height
        n_dom = filtered.select('domain').n_unique()
    
    n_alive = int(n_cand * URL_ALIVE_RATE)
    n_dl = int(n_alive * DOWNLOAD_RATE)
    n_pass = int(n_dl * AI_PASS_RATE)
    cost = n_dl * COST_PER_IMAGE
    pct = cost / BUDGET_FORMAL * 100
    
    print(f'{name:<25} {n_cand:>12,} {n_dom:>8,} {n_alive:>11,} {n_dl:>16,} {n_pass:>10,} {f"${cost:,.0f}":>8} {pct:>8.0f}%')

After exclusions: 1,330,153 (excluded 25,659)

Scenario                    Candidates  Domains  Est. alive  Est. downloaded  Est. maps     Cost  % budget
--------------------------------------------------------------------------------------------------------------
A: p>=0.70, cap 50             424,990  123,280     276,243          267,955    200,966     $536       30%
B: p>=0.70, cap 100            477,285  123,280     310,235          300,927    225,695     $602       33%
C: p>=0.60, cap 100            544,427  138,122     353,877          343,260    257,445     $687       38%
D: p>=0.50, cap 100            612,700  153,307     398,255          386,307    289,730     $773       43%
E: p>=0.70, cap 200            528,053  123,280     343,234          332,936    249,702     $666       37%
F: p>=0.50, cap 50             542,821  153,307     352,833          342,248    256,686     $684       38%
G: p>=0.80, cap 100            401,922  106,382     261,249          253,411    190,058     $

## 5. Impact of domain cap on diversity

Ile obrazów tracimy z dużych domen przy różnych capach?

In [13]:
# For pred_proba >= 0.70
filtered = df_clean.filter(pl.col('pred_proba') >= 0.70)
dom_sizes = filtered.group_by('domain').agg(pl.len().alias('n')).sort('n', descending=True)

print('=== DOMAIN CAP IMPACT (pred_proba >= 0.70) ===')
print(f'  No cap: {filtered.height:,} candidates from {dom_sizes.height:,} domains')
print()

for cap in [10, 25, 50, 100, 200, 500]:
    capped_total = dom_sizes.with_columns(
        pl.when(pl.col('n') > cap).then(cap).otherwise(pl.col('n')).alias('capped')
    )['capped'].sum()
    lost = filtered.height - capped_total
    affected = dom_sizes.filter(pl.col('n') > cap).height
    print(f'  cap={cap:>3}: {capped_total:>10,} candidates ({affected:,} domains capped, {lost:,} images lost)')

=== DOMAIN CAP IMPACT (pred_proba >= 0.70) ===
  No cap: 1,048,512 candidates from 123,280 domains

  cap= 10:    299,309 candidates (7,453 domains capped, 749,203 images lost)
  cap= 25:    370,887 candidates (3,049 domains capped, 677,625 images lost)
  cap= 50:    424,990 candidates (1,524 domains capped, 623,522 images lost)
  cap=100:    477,285 candidates (730 domains capped, 571,227 images lost)
  cap=200:    528,053 candidates (355 domains capped, 520,459 images lost)
  cap=500:    590,236 candidates (125 domains capped, 458,276 images lost)


## 6. Domains to investigate

Duże domeny z niskim mean_proba mogą być śmieciowe. Duże domeny z wysokim -- mogą być wartościowe ale redundantne.

In [14]:
# Large domains with LOW mean_proba (suspicious -- might not be statistical maps)
print('=== LARGE DOMAINS WITH LOW MEAN PROBA (possible junk) ===')
suspicious = domains.filter((pl.col('count') > 100) & (pl.col('mean_proba') < 0.75))
for row in suspicious.head(20).iter_rows(named=True):
    print(f'  {row["domain"]:<45}  n={row["count"]:>6,}  mean_p={row["mean_proba"]:.3f}')

print()
print('=== LARGE DOMAINS WITH HIGH MEAN PROBA (likely good but redundant) ===')
good_big = domains.filter((pl.col('count') > 500) & (pl.col('mean_proba') > 0.85))
for row in good_big.head(20).iter_rows(named=True):
    print(f'  {row["domain"]:<45}  n={row["count"]:>6,}  mean_p={row["mean_proba"]:.3f}')

=== LARGE DOMAINS WITH LOW MEAN PROBA (possible junk) ===
  files.airnowtech.org                           n=11,516  mean_p=0.711
  img.freepik.com                                n= 1,772  mean_p=0.734
  storage.tenki.jp                               n= 1,100  mean_p=0.713
  img.ltn.com.tw                                 n=   893  mean_p=0.734
  www.nohrsc.noaa.gov                            n=   801  mean_p=0.668
  images.slideplayer.nl                          n=   671  mean_p=0.741
  dr5dymrsxhdzh.cloudfront.net                   n=   607  mean_p=0.713
  cdn-2.tstatic.net                              n=   573  mean_p=0.748
  makeshop-multi-images.akamaized.net            n=   535  mean_p=0.644
  resources.mynewsdesk.com                       n=   499  mean_p=0.735
  a57.foxnews.com                                n=   453  mean_p=0.744
  acp.copernicus.org                             n=   410  mean_p=0.734
  www.severe-weather.eu                          n=   402  mean_p=0.740
  www.

## 7. Final decision

Edytuj parametry i uruchom ponownie.

In [16]:
# === FINAL PARAMETERS ===
PRED_PROBA_MIN = 0.70    # adjust after analysis
DOMAIN_CAP = 50          # adjust after analysis

# Apply
final = df_clean.filter(pl.col('pred_proba') >= PRED_PROBA_MIN)
final_capped = final.group_by('domain').agg(
    pl.len().alias('n')
).with_columns(
    pl.when(pl.col('n') > DOMAIN_CAP).then(DOMAIN_CAP).otherwise(pl.col('n')).alias('capped')
)

n_candidates = final_capped['capped'].sum()
n_domains = final_capped.height
n_alive = int(n_candidates * URL_ALIVE_RATE)
n_downloaded = int(n_alive * DOWNLOAD_RATE)
n_maps = int(n_downloaded * AI_PASS_RATE)
cost = n_downloaded * COST_PER_IMAGE

print(f'=== FINAL SAMPLING STRATEGY ===')
print(f'  pred_proba >= {PRED_PROBA_MIN}')
print(f'  domain_cap = {DOMAIN_CAP}')
print(f'  excluded domains: {len(EXCLUDE_DOMAINS)}')
print(f'  URL deduplication: yes')
print()
print(f'  Candidates to sample: {n_candidates:,}')
print(f'  Unique domains: {n_domains:,}')
print(f'  Est. URL alive: {n_alive:,}')
print(f'  Est. downloaded: {n_downloaded:,}')
print(f'  Est. AI pass (maps in dataset): {n_maps:,}')
print(f'  Est. API cost: ${cost:,.0f}')
print(f'  Budget remaining: ${BUDGET_FORMAL - cost:,.0f}')

=== FINAL SAMPLING STRATEGY ===
  pred_proba >= 0.7
  domain_cap = 50
  excluded domains: 3
  URL deduplication: yes

  Candidates to sample: 424,990
  Unique domains: 123,280
  Est. URL alive: 276,243
  Est. downloaded: 267,955
  Est. AI pass (maps in dataset): 200,966
  Est. API cost: $536
  Budget remaining: $1,264
